In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("hf_token_add")

In [2]:
import datasets
from huggingface_hub import HfApi

api = HfApi(token=secret_value_0)

files = api.list_repo_files(
    repo_id="stukenov/sozkz-corpus-clean-v3",
    repo_type="dataset"
)

In [3]:
files = files[2:].copy()

In [4]:
SOURCES = {'belebele',
 'cc100',
 'culturax',
 'hplt_new',
 'kazparc',
 'kazparc_sync',
 'kazsandra',
 'madlad400',
 'mc4',
 'md_kazakhBooks',
 'md_kazakhNews',
 'md_leipzig',
 'md_oscar',
 'moscar',
 'sib200',
 'wikiann',
 'wikipedia'}

In [5]:
import sys
sys.path.insert(0, "/kaggle/input/datasets/salyamq/cleaning")

from cleaning import clean_document, is_spam
from counting import init_stats, clean_document_with_stats, save_stats, load_stats

In [6]:
STATS_PATH = "/kaggle/working/cleaning_stats.json"
SRC_DATASET    = "stukenov/sozkz-corpus-clean-v3"   
DST_DATASET    = "salyamq/kz_ds_v1"  
SPAM_PATH   = "/kaggle/working/spam_docs.jsonl"
START_FROM     = 142

In [7]:
import os
import gc
import json
import tempfile
from datasets import load_dataset
from huggingface_hub import HfApi

shards = files

api = HfApi(token=secret_value_1)

if os.path.exists(STATS_PATH):
    stats = load_stats(STATS_PATH)
else:
    stats = init_stats(SOURCES)

shards_to_process = shards[START_FROM:]
print(f"shards: {len(shards_to_process)}")

for i, shard_file in enumerate(shards_to_process):
    global_idx = START_FROM + i
    print(f"── [{global_idx+1}/{len(shards)}] {shard_file} ──")

    ds = load_dataset(
        SRC_DATASET,
        data_files=shard_file,
        split="train",
        token=secret_value_0,
        verification_mode="no_checks"
    )
    print(f"documents: {len(ds):,}")

    spam_buffer = []

    def clean_row(example):
        original = example["text"]
        cleaned = clean_document_with_stats(original, example["source"], stats)
        if cleaned is None and isinstance(original, str) and is_spam(original):
            spam_buffer.append({"text": original, "source": example["source"]})
        example["text"] = cleaned
        return example

    ds = ds.map(clean_row, desc="clean", remove_columns=[])

    before = len(ds)
    ds = ds.filter(lambda x: x["text"] is not None)
    after = len(ds)
    pct = (before - after) / before * 100 if before > 0 else 0
    print(f"discarded: {before - after:,} ({pct:.1f}%) - left: {after:,}")


    shard_name = os.path.basename(shard_file)
    with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
        tmp_path = tmp.name

    ds.to_parquet(tmp_path)
    api.upload_file(
        path_or_fileobj=tmp_path,
        path_in_repo=f"data/{shard_name}",
        repo_id=DST_DATASET,
        repo_type="dataset",
        commit_message=f"add {shard_name}",
    )
    os.remove(tmp_path)
    print(f"uploaded: data/{shard_name}")

    save_stats(stats, STATS_PATH)

    if spam_buffer:
        with open(SPAM_PATH, "a", encoding="utf-8") as f:
            for doc in spam_buffer:
                f.write(json.dumps(doc, ensure_ascii=False) + "\n")
        print(f"spam saved: {len(spam_buffer)}")

    ds.cleanup_cache_files()
    del ds
    gc.collect()

shards: 128
── [143/270] data/shard_00143.parquet ──


README.md: 0.00B [00:00, ?B/s]

data/shard_00143.parquet:   0%|          | 0.00/129M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13463018 [00:00<?, ? examples/s]

documents: 50,000


...

discarded: 372 (0.9%) - left: 40,381


Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

uploaded: data/shard_00274.parquet
spam saved: 367
